# This notebook is to sort by channel and then cluster each channel

**USE Cleaned Data!**

In [27]:
from treeview_gen import df_to_treeview
import pandas as pd
import os
import re
import shutil
from time import sleep

regex_df = pd.read_csv("/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/cleaned_data/feature_reduction_cleaned/HistDiff_results_24h_bmal1ref_FeatRed_WELL-DROPPED.csv", index_col=0, nrows=1)
pattern = r'Region (.+?) \(global\)'
channels_found = set()
for col in regex_df.columns:
    found = re.search(pattern, col)
    if found:
        channels_found.add(found.group(1))

print(channels_found)


{'SYTO14 Rabbitts', 'MitoTracker Deep Red', 'HOECHST 33342 expanded', 'TMR/AF555 Rabbitts', 'Alexa 488 shortened'}


In [54]:
def gen_treeview_channel(df: pd.DataFrame, channel_name: str, out_dir: str) -> list[str]:
    """Generates a treeviewe file set for a given channel

    Args:
        df (pd.DataFrame): channel data frame
        channel_name (str): the name of the channel
        out_dir (str): the file path where the file will be outputted to (this creates a folder)
    """
    
    out_dir_channel = os.path.join(out_dir, channel_name)
    
    if os.path.exists(out_dir_channel):
        shutil.rmtree(out_dir_channel)
        sleep(0.5)
    os.makedirs(out_dir_channel, exist_ok=True)
    
    out_dir_channel = os.path.join(out_dir_channel, f"{channel_name.replace("/","_")}_treeview")
    
    df_to_treeview(df=df, outFile=out_dir_channel, sortIndex=True, row_cluster=False)
    
    cdt_path = f"{out_dir_channel}.cdt"
    return read_tv_col_order(cdt_path)


def read_tv_col_order(file_path: str) -> list[str]:
    col_df = pd.read_csv(file_path, index_col=[0,1,2], sep="\t")
    return col_df.columns.tolist()

In [52]:
test_cdt = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/Heatmaps (treeview)/channel sorted/HistDiff_results_24h_bmal1ref_FeatRed_WELL-DROPPED/Alexa 488 shortened/Alexa 488 shortened_treeview.cdt"
cdt_df = pd.read_csv(test_cdt, sep="\t", index_col=[0,1,2])

print(cdt_df.columns.tolist())

['Nuclei - Ring Region Alexa 488 shortened (global) Symmetry 02 SER-Saddle', 'Nuclei - Ring Region Alexa 488 shortened (global) Symmetry 02 SER-Edge', 'Nuclei - Ring Region Alexa 488 shortened (global) Symmetry 13 SER-Edge', 'Nuclei - Cytoplasm Region Alexa 488 shortened (global) Symmetry 12 SER-Saddle', 'Nuclei - Cytoplasm Region Alexa 488 shortened (global) Symmetry 02 SER-Saddle', 'Nuclei - Cytoplasm Region Alexa 488 shortened (global) Symmetry 15 SER-Saddle', 'Nuclei - Cytoplasm Region Alexa 488 shortened (global) Symmetry 05 SER-Saddle', 'Nuclei - Membrane Region Alexa 488 shortened (global) Symmetry 05 SER-Saddle', 'Nuclei - Cell Region Alexa 488 shortened (global) Symmetry 05 SER-Saddle', 'Nuclei - Cell Region Alexa 488 shortened (global) Symmetry 15 SER-Saddle', 'Nuclei - Cytoplasm Region Alexa 488 shortened (global) Symmetry 03 SER-Saddle', 'Nuclei - Cytoplasm Region Alexa 488 shortened (global) Symmetry 13 SER-Saddle', 'Nuclei - Cell Region Alexa 488 shortened (global) Symmet

## Final Loop

In [ ]:
cleaned_file_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/cleaned_data/feature_reduction_cleaned"
out_dir = "/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/Heatmaps (treeview)/channel sorted"

for file in os.listdir(cleaned_file_dir):
    curr_file = os.path.join(cleaned_file_dir, file)
    print(curr_file)
    raw_name = os.path.splitext(file)[0]
    print(raw_name)
    
    raw_out_dir = os.path.join(out_dir, raw_name)
    print(raw_out_dir)
    
    # make a folder
    if os.path.exists(raw_out_dir):
        shutil.rmtree(raw_out_dir)
        sleep(0.5)
    os.makedirs(raw_out_dir)
    
    
    curr_df = pd.read_csv(curr_file, index_col=0)
    
    # grab channels from data
    channel_df = {ch: curr_df[[col for col in curr_df.columns if ch in col]] for ch in channels_found}
    
    # grab the column orders and export out the channel clustering
    col_order = []
    for name, df in channel_df.items():
        order = gen_treeview_channel(df, name.replace("/", "_"), raw_out_dir)
        col_order.extend(order)
    
    print(col_order)
    
    reorder_df = curr_df[col_order]
    
    final_out_path = os.path.join(raw_out_dir, f"{raw_name}_SORTED-CLUSTERED_channels")
    df_to_treeview(df=reorder_df, outFile=final_out_path, sortIndex=True, row_cluster=False, col_cluster=False)

/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/cleaned_data/feature_reduction_cleaned/HistDiff_results_24h_bmal1ref_FeatRed_WELL-DROPPED.csv
HistDiff_results_24h_bmal1ref_FeatRed_WELL-DROPPED
/home/derfelt/PartchLabFiles/Circadian Rhythms/Pilot Study/Heatmaps (treeview)/channel sorted/HistDiff_results_24h_bmal1ref_FeatRed_WELL-DROPPED
['Nuclei - Membrane Region SYTO14 Rabbitts (global) Symmetry 15 SER-Spot', 'Nuclei - Membrane Region SYTO14 Rabbitts (global) Symmetry 13 SER-Ridge', 'Nuclei - Ring Region SYTO14 Rabbitts (global) Symmetry 03 SER-Saddle', 'Nuclei - Ring Region SYTO14 Rabbitts (global) Symmetry 05 SER-Saddle', 'Nuclei - Membrane Region SYTO14 Rabbitts (global) Symmetry 04 SER-Edge', 'Nuclei - Membrane Region SYTO14 Rabbitts (global) Symmetry 04 SER-Saddle', 'Nuclei - Membrane Region SYTO14 Rabbitts (global) Symmetry 12 SER-Hole', 'Nuclei - Cytoplasm Region SYTO14 Rabbitts (global) Symmetry 12 SER-Hole', 'Nuclei - Cytoplasm Region SYTO14 Rabbitts (global) Symmet